# Foucault test data reduction

Turns Foucault / wire test readings (knife position vs. nulled zone radius) into the numbers needed while figuring a mirror:

* **best-fit conic constant** and **% correction** against the target conic,
* **wavefront error** relative to the target conic at best focus, as rotationally symmetric **Zernike terms in Fringe/DFTFringe numbering** (Z3, Z8, Z15, Z24), so results can be compared directly with Bath interferometer reductions in DFTFringe,
* **RMS, P-V and Strehl** with **uncertainty intervals** derived from the scatter of the data,
* a **surface error profile in nm** (+ = high, i.e. glass to remove) to guide the next figuring session.

### Relation to the established methods

* *Classical reduction* (Couder mask + Texereau tables, FigureXP, Sixtests): readings are converted to transverse aberration / slope error per zone and integrated zone by zone. Done here as a cross-check ("zone-by-zone integration").
* *Modal slope integration*: the slope errors are fitted directly with the derivatives of Zernike polynomials (standard practice for slope-measuring tests such as Shack-Hartmann and deflectometry). This is linear, smooth, gives parameter covariances, and absorbs the unknown zero of the knife scale and the error of the measured $R$ into defocus automatically.
* Everything is computed **as a deviation from the target conic**: the quantities of interest (~10 nm) are about five orders of magnitude smaller than the mirror sag (~2 mm), so integrating the full sag (as in the earlier exploratory draft) wastes almost all numerical precision.

Limitations of the Foucault test itself: it only sees the rotationally symmetric part of the figure along the measured diameter. Astigmatism and other non-symmetric errors must come from the Bath test.

## 1. Method

**Target surface.** A conic of vertex radius $R$ and conic constant $k$ (ISO 10110-12):

$$z(r) = \frac{r^2}{R\left(1+\sqrt{1-(1+k)\,r^2/R^2}\right)}$$

Its normal at radius $r$ crosses the optical axis at

$$Z(r) = z(r) + \frac{r}{z'(r)} = z(r) + \sqrt{R^2-(1+k)\,r^2}$$

(for a paraboloid $Z - R = r^2/2R$).

**Readings.** With the source moving together with the knife, the knife nulls zone $r$ at its normal intercept: $Z_m = R + s$. With a fixed source the longitudinal aberration is doubled, so $Z_m \approx R + s/2$. The unknown zero of the knife scale is absorbed later.

**Slope error.** From $z'(r) = r / (Z - z)$ (the same relation as the ODE of the draft), the surface slope error relative to the target is

$$\delta z'(r) = \frac{r}{Z_m-z_t} - \frac{r}{Z_t-z_t} = -\frac{r\,(Z_m - Z_t)}{(Z_m - z_t)(Z_t - z_t)}, \qquad W'(r) = 2\,\delta z'(r)$$

where $W$ is the wavefront error (reflection doubles it). A constant offset $c$ of the knife scale or an error in $R$ adds $\approx -rc/R^2$ to the slope, i.e. pure defocus.

**Modal fit.** With $\rho = r/a$ ($a$ = clear aperture radius) and $W(\rho) = \sum_j a_j Z_j(\rho)$:

$$W'(r_i) = \frac{1}{a}\sum_j a_j\,\frac{dZ_j}{d\rho}\!\left(\frac{r_i}{a}\right)$$

is solved by linear least squares. Z3 (defocus) absorbs the knife zero offset and the $R$ error.

**Figures of merit** (after area-weighted removal of piston and defocus, i.e. at best focus):

$$\mathrm{RMS}^2 = \frac{1}{A}\int W^2\,dA, \qquad S = \left|\frac{1}{A}\int e^{\,i 2\pi W/\lambda}\,dA\right|^2, \qquad dA = 2\pi\rho\,d\rho$$

The Strehl ratio is computed by direct integration, so it does not rely on the Maréchal approximation.

## 2. Configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from numpy.polynomial import polynomial as P
from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.optimize import least_squares

plt.rcParams['figure.figsize'] = [9, 4]
plt.rcParams['figure.dpi'] = 110

DATA_FILE = Path('mirror_measurements_database.json')
WAVELENGTH_NM = 550.0
TARGET_K = -1.0                 # desired conic constant (paraboloid = -1)
SOURCE_MOVES_WITH_KNIFE = True  # False if the light source stays fixed while the knife moves
CLEAR_APERTURE_RADIUS = None    # mm, None = diameter / 2
OBSTRUCTION_RADIUS = 0.0        # mm, secondary mirror shadow, excluded from the statistics
ZERNIKE_TERMS = (3, 8, 15)      # Fringe (DFTFringe) numbering; 3 is mandatory, add 24 only with dense data
N_MONTE_CARLO = 2000

## 3. Load the test data

In [ ]:
with DATA_FILE.open() as fh:
    data_base = json.load(fh)

diameter = data_base['diameter']
R = data_base['radius_of_curvature_measurement']['radius_of_curvature']
a = CLEAR_APERTURE_RADIUS or diameter / 2
wavelength = WAVELENGTH_NM * 1e-6  # mm

measurements = pd.DataFrame(data_base['measurements']).sort_values('mirror_radius', ignore_index=True)
r = measurements['mirror_radius'].to_numpy(float)
readings = measurements['optical_axis_offset'].to_numpy(float)

print(f"{data_base['mirror_id']} ({data_base['description']}, {data_base['date']})")
print(f"D = {diameter} mm, R = {R} mm, f/{R / 2 / diameter:.2f}, {len(r)} zones from {r.min()} to {r.max()} mm")
display(measurements[['mirror_radius', 'optical_axis_offset']])

## 4. Reduction functions

In [ ]:
FRINGE_RADIAL = {  # rotationally symmetric Fringe Zernike polynomials, ascending powers of rho
    3: [-1, 0, 2],
    8: [1, 0, -6, 0, 6],
    15: [-1, 0, 12, 0, -30, 0, 20],
    24: [1, 0, -20, 0, 90, 0, -140, 0, 70],
}
ZERNIKE_NAMES = {3: 'defocus', 8: 'spherical', 15: 'secondary spherical', 24: 'tertiary spherical'}


def conic_sag(r, R, k):
    return r**2 / (R * (1 + np.sqrt(1 - (1 + k) * r**2 / R**2)))


def conic_normal_intercept(r, R, k):
    """Distance from the vertex to where the surface normal at radius r crosses the optical axis."""
    return conic_sag(r, R, k) + np.sqrt(R**2 - (1 + k) * r**2)


def normal_intercept_from_readings(readings, R, source_moves):
    # A fixed source doubles the longitudinal aberration (first order).
    return R + (readings if source_moves else readings / 2)


def fit_conic(r, readings, R, source_moves):
    """Best-fit conic constant; the unknown zero of the knife scale is a free parameter."""
    Zm = normal_intercept_from_readings(readings, R, source_moves)
    sol = least_squares(lambda p: conic_normal_intercept(r, R, p[1]) + p[0] - Zm, x0=[0.0, -1.0])
    cov = (sol.fun @ sol.fun / (len(r) - 2)) * np.linalg.inv(sol.jac.T @ sol.jac)
    return sol.x, cov, sol.fun


def wavefront_slopes(r, readings, R, target_k, source_moves):
    """Wavefront slope error [rad] relative to the target conic at each measured zone."""
    Zm = normal_intercept_from_readings(readings, R, source_moves)
    zt = conic_sag(r, R, target_k)
    Zt = conic_normal_intercept(r, R, target_k)
    return -2 * r * (Zm - Zt) / ((Zm - zt) * (Zt - zt))


def zernike_slope_matrix(r, a, terms):
    return np.column_stack([P.polyval(r / a, P.polyder(FRINGE_RADIAL[j])) / a for j in terms])


def fit_zernikes(r, slopes, a, terms):
    A = zernike_slope_matrix(r, a, terms)
    coef, *_ = np.linalg.lstsq(A, slopes, rcond=None)
    residual = slopes - A @ coef
    cov = (residual @ residual / max(len(r) - len(terms), 1)) * np.linalg.inv(A.T @ A)
    return coef, cov, A @ coef


def remove_piston_defocus(rho, W):
    """Area-weighted best-fit piston + defocus removal along the last axis (= best focus)."""
    B = np.column_stack([np.ones_like(rho), rho**2])
    w = np.sqrt(rho)
    fit, *_ = np.linalg.lstsq(B * w[:, None], (np.atleast_2d(W) * w).T, rcond=None)
    return W - (B @ fit).T.reshape(W.shape)


def pupil_basis(rho, terms):
    """Zernike terms sampled on rho, each with piston and defocus over that pupil removed."""
    return np.column_stack([remove_piston_defocus(rho, P.polyval(rho, FRINGE_RADIAL[j])) for j in terms])


def figures_of_merit(rho, W, wavelength):
    """RMS, P-V and Strehl of wavefront(s) W sampled on rho (last axis), piston/defocus removed."""
    area = trapezoid(rho, rho)
    rms = np.sqrt(trapezoid(W**2 * rho, rho, axis=-1) / area)
    pv = W.max(axis=-1) - W.min(axis=-1)
    strehl = np.abs(trapezoid(np.exp(2j * np.pi * W / wavelength) * rho, rho, axis=-1) / area)**2
    return rms, pv, strehl

### 4.1. Self-check with synthetic data

Readings are generated for a perfect conic with a known $k$ and an arbitrary knife zero offset, at the same zone radii as the real data. The reduction must recover $k$ and the exact wavefront difference to the target conic.

In [ ]:
k_true = -0.85
check_terms = (3, 8, 15, 24)
check_readings = conic_normal_intercept(r, R, k_true) - R + 0.37
if not SOURCE_MOVES_WITH_KNIFE:
    check_readings = 2 * check_readings

(_, k_check), _, _ = fit_conic(r, check_readings, R, SOURCE_MOVES_WITH_KNIFE)
coef_check, _, _ = fit_zernikes(r, wavefront_slopes(r, check_readings, R, TARGET_K, SOURCE_MOVES_WITH_KNIFE), a, check_terms)

rho_check = np.linspace(0, 1, 2001)
W_reconstructed = pupil_basis(rho_check, check_terms) @ coef_check
W_exact = remove_piston_defocus(rho_check, 2 * (conic_sag(rho_check * a, R, k_true) - conic_sag(rho_check * a, R, TARGET_K)))
print(f"k recovered: {k_check:.4f} (true {k_true})")
print(f"max wavefront reconstruction error: {np.abs(W_reconstructed - W_exact).max() * 1e6:.3f} nm "
      f"(P-V of the true wavefront error: {np.ptp(W_exact) * 1e6:.1f} nm)")

## 5. Best-fit conic and correction

Classic view of the test: measured normal-intercept shifts compared with the target and best-fit conics. Points systematically between the two curves at some zones indicate zonal errors rather than a wrong conic.

In [ ]:
(c_fit, k_fit), conic_cov, conic_residual = fit_conic(r, readings, R, SOURCE_MOVES_WITH_KNIFE)
k_sigma = np.sqrt(conic_cov[1, 1])
print(f"Best-fit conic constant k = {k_fit:.3f} ± {k_sigma:.3f} (1σ)")
if TARGET_K != 0:
    print(f"Correction = {100 * k_fit / TARGET_K:.1f} % ± {100 * k_sigma / abs(TARGET_K):.1f} % of target k = {TARGET_K}")
print(f"RMS reading residual about the best-fit conic: {np.sqrt(np.mean(conic_residual**2)):.3f} mm")

Zm = normal_intercept_from_readings(readings, R, SOURCE_MOVES_WITH_KNIFE)
rr = np.linspace(0, a, 300)
fig, ax = plt.subplots()
ax.plot(r, Zm - R - c_fit, 'o', label='Measured (knife zero offset fitted)')
ax.plot(rr, conic_normal_intercept(rr, R, k_fit) - R, label=f'Best-fit conic, k = {k_fit:.3f}')
ax.plot(rr, conic_normal_intercept(rr, R, TARGET_K) - R, '--', label=f'Target conic, k = {TARGET_K}')
ax.set(title='Longitudinal aberration (normal intercept shift)', xlabel='Zone radius [mm]', ylabel='Shift [mm]')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 6. Wavefront error relative to the target conic

Zernike coefficients are in waves of wavefront, Fringe numbering as in DFTFringe, relative to the target conic. Compare Z8 with the Bath result reduced with the same desired conic. Coefficients whose 1σ uncertainty is comparable to their value are not supported by the data. Structure in the slope residuals indicates zonal errors that the chosen terms cannot follow.

In [ ]:
slopes = wavefront_slopes(r, readings, R, TARGET_K, SOURCE_MOVES_WITH_KNIFE)
coef, coef_cov, slopes_fit = fit_zernikes(r, slopes, a, ZERNIKE_TERMS)

display(pd.DataFrame({
    'term': [f'Z{j}' for j in ZERNIKE_TERMS],
    'name': [ZERNIKE_NAMES[j] for j in ZERNIKE_TERMS],
    'waves': coef / wavelength,
    '± 1σ waves': np.sqrt(np.diag(coef_cov)) / wavelength,
}).round(4))

fig, (ax0, ax1) = plt.subplots(2, 1, sharex=True, figsize=(9, 6), gridspec_kw={'height_ratios': [2, 1]})
ax0.plot(r, slopes * 1e6, 'o', label='Measured')
ax0.plot(rr, zernike_slope_matrix(rr, a, ZERNIKE_TERMS) @ coef * 1e6, label='Zernike fit')
ax0.set(title='Wavefront slope error relative to the target conic', ylabel='Slope [µrad]')
ax0.legend(); ax0.grid(alpha=0.3)
ax1.plot(r, (slopes - slopes_fit) * 1e6, 'o-')
ax1.axhline(0, color='k', lw=0.8)
ax1.set(xlabel='Zone radius [mm]', ylabel='Residual [µrad]'); ax1.grid(alpha=0.3)
plt.show()

## 7. Figures of merit

Uncertainty intervals come from Monte Carlo sampling of the Zernike coefficients with the covariance estimated from the fit residuals. They cover random measurement scatter only. Systematic errors are not included, for example a wrong source configuration, a zone radius scale error, or a knife that is not on the axis.

Statistics are given both for the full pupil and for the measured annulus only. Outside the measured zones the Zernike model is extrapolating. The mirror edge is where P-V is usually decided, so do not trust full-pupil P-V unless zones close to the edge were measured.

In [ ]:
rng = np.random.default_rng(0)
coef_samples = rng.multivariate_normal(coef, coef_cov, N_MONTE_CARLO)

pupils = {
    'full pupil': np.linspace(OBSTRUCTION_RADIUS / a, 1, 2001),
    'measured annulus': np.linspace(max(OBSTRUCTION_RADIUS, r.min()) / a, r.max() / a, 2001),
}
rows = {}
for name, rho_p in pupils.items():
    basis = pupil_basis(rho_p, ZERNIKE_TERMS)
    nominal = figures_of_merit(rho_p, basis @ coef, wavelength)
    sampled = figures_of_merit(rho_p, coef_samples @ basis.T, wavelength)
    for metric, scale, value, samples in zip(('RMS [waves]', 'P-V [waves]', 'Strehl'), (1 / wavelength, 1 / wavelength, 1), nominal, sampled):
        lo, hi = np.percentile(samples * scale, [5, 95])
        rows.setdefault(metric, {})[name] = f'{value * scale:.3f}  [{lo:.3f} … {hi:.3f}]'
    if name == 'full pupil':
        rho, W, W_mc = rho_p, basis @ coef, coef_samples @ basis.T

summary = pd.DataFrame(rows).T
summary['usual goal'] = ['≤ 0.071 (λ/14)', '≤ 0.25 (λ/4)', '≥ 0.80']
print(f'Wavefront at {WAVELENGTH_NM:.0f} nm, best focus, relative to k = {TARGET_K}; value [90 % interval]')
display(summary)

area_fraction = (min(r.max(), a)**2 - max(r.min(), OBSTRUCTION_RADIUS)**2) / (a**2 - OBSTRUCTION_RADIUS**2)
print(f'Measured zones cover {100 * area_fraction:.0f} % of the evaluated pupil area.')

## 8. Surface error profile (figuring guide)

Surface error is half the wavefront error. It is measured relative to the target conic at best focus. Positive values are high zones (excess glass). The zone-by-zone points come from direct trapezoidal integration of the measured slopes, with no model involved. Where they depart consistently from the Zernike curve, the mirror has zonal structure that the chosen terms cannot represent. Grey regions were not measured.

In [ ]:
W_model_at_zones = sum(c * P.polyval(r / a, FRINGE_RADIAL[j]) for c, j in zip(coef, ZERNIKE_TERMS))
deviation = cumulative_trapezoid(slopes, r, initial=0.0) - W_model_at_zones
W_zonal = np.interp(r / a, rho, W) + deviation - deviation.mean()

to_surface_nm = 1e6 / 2
lo, hi = np.percentile(W_mc, [5, 95], axis=0)
fig, ax = plt.subplots()
for side in (1, -1):
    first = side == 1
    ax.fill_between(side * rho * a, lo * to_surface_nm, hi * to_surface_nm, color='C0', alpha=0.2, label='90 % interval' if first else None)
    ax.plot(side * rho * a, W * to_surface_nm, 'C0', label='Zernike reconstruction' if first else None)
    ax.plot(side * r, W_zonal * to_surface_nm, 'C1.', label='Zone-by-zone integration' if first else None)
    ax.axvspan(side * r.max(), side * a, color='grey', alpha=0.15, label='Not measured' if first else None)
ax.axvspan(-r.min(), r.min(), color='grey', alpha=0.15)
ax.axhline(0, color='k', lw=0.8)
ax.set(title=f'Surface error vs. target conic k = {TARGET_K} at best focus (+ = high)',
       xlabel='Mirror radius [mm]', ylabel='Surface error [nm]')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## 9. Recommendations for data acquisition

* **Record the tester configuration in the JSON file**: moving or fixed source, wavelength/filter, and the uncertainty of the $R$ measurement. A wrong source assumption halves or doubles every aberration.
* **Measure both sides of the mirror** at each knife position and average the left and right radii. This cancels a knife or source that is laterally off-axis and exposes asymmetries. The current data has one radius per reading.
* **Extend coverage** to about 0.2 a near the centre and 0.95–0.97 a near the edge. A turned edge only shows up there, and without it full-pupil P-V/Strehl are extrapolated.
* **Use the motorised stage and camera for dense scans**: step the knife finely, and for every pixel along the horizontal diameter find the knife position where the intensity crosses the midpoint between its dark and bright levels (the null). This gives hundreds of zones on both sides with the same `r`/`readings` arrays, so this notebook works unchanged and higher Zernike terms (Z24) become meaningful.
* **Repeat runs** (3 or more) and reduce them together. Scatter between runs is the honest noise estimate.
* **Cross-check with Bath/DFTFringe**: Z8 and % correction should agree within the intervals above. Astigmatism and non-symmetric errors come from Bath only.